# 📊 Decile Churn Analysis — Revenue Decomposition
**LushProtein Analytics | Medallion Architecture**

Reads from `gold_churn_features` and `gold_customer_profiles`, assigns customers to revenue deciles, computes per-decile retention and behavioural metrics, and visualises revenue concentration, AOV/AOF patterns, 90-day repeat rates, and discount/subscriber profiles.

| Source Tables | Key Analyses |
|---|---|
| `gold_churn_features`, `gold_customer_profiles` | Revenue decile assignment, repeat rate by decile, AOV & AOF decomposition, discount acquisition & subscriber rate by decile |

## 0. Setup & Paths

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 80)

BASE       = Path.cwd()
SILVER_DIR = BASE / 'medallion' / 'silver'
GOLD_DIR   = BASE / 'medallion' / 'gold'

sns.set_theme(style='whitegrid', palette='muted')
COLORS = {
    'primary'  : '#2E86AB',
    'secondary': '#E84855',
    'accent'   : '#3BB273',
    'neutral'  : '#A8A8A8',
    'orange'   : '#F97316',
}

---
## 1. Load Data

### Step 1 — Load `gold_churn_features` & `gold_customer_profiles`

In [4]:
# Load gold_customer_profiles 
cp = pd.read_parquet(GOLD_DIR / 'gold_customer_profiles.parquet')
df = pd.read_parquet(GOLD_DIR / 'gold_churn_features.parquet')

---
## 2. Build Analysis Base

### Step 2 — Merge profiles and featuers & Derive AOV / AOF

In [14]:
# Merge both customer profiles and customer features
analysis = df.merge(
    cp[['customer_id', 'total_revenue']],
    on='customer_id', how='left'
)

# Flag rather than exclude zero-price first orders
analysis['first_order_is_zero_price'] = analysis['first_order_is_zero_price'].fillna(False)

# Keep only customers with total_revenue > 0
analysis = analysis[
    analysis['total_revenue'].notna() &
    (analysis['total_revenue'] > 0)
].copy()

# Derive AOF (repeat orders = total orders - 1)
analysis['aof'] = analysis['total_orders'] - 1

# Derive AOV (total revenue / total orders)
analysis['aov'] = (analysis['total_revenue'] / analysis['total_orders']).round(2)

print(f"Analysis base: {len(analysis):,} customers")

# Flag and report zero-price first order customers
zero_price_total    = analysis['first_order_is_zero_price'].sum()
zero_price_pct      = analysis['first_order_is_zero_price'].mean()
converted           = analysis[analysis['first_order_is_zero_price'] == True]
pure_zero           = df[
    (df['first_order_is_zero_price'] == True) &
    (~df['customer_id'].isin(analysis['customer_id']))
]

print(f"\n=== Zero-Price First Order Customers ===")
print(f"Retained in analysis (have subsequent paid orders) : {zero_price_total:,} ({zero_price_pct:.1%})")
print(f"Excluded (never paid anything)                     : {len(pure_zero):,}")

Analysis base: 12,938 customers

=== Zero-Price First Order Customers ===
Retained in analysis (have subsequent paid orders) : 41 (0.3%)
Excluded (never paid anything)                     : 777


---
## 3. Decile Assignment & Metrics

### Step 3 — Assign revenue deciles & compute per-decile metrics

In [22]:
bins = [0, 25, 50, 75, 100, 150, 200, 350, 600, 1500, float('inf')]
labels = [
    'D10: <25',
    'D9: 25–50',
    'D8: 50–75',
    'D7: 75–100',
    'D6: 100–150',
    'D5: 150–200',
    'D4: 200–350',
    'D3: 350–600',
    'D2: 600–1500',
    'D1: >1500',
]

analysis['vtd_tier'] = pd.cut(
    analysis['total_revenue'],
    bins=bins,
    labels=labels
)

# Compute metrics per tier
tier_metrics = analysis.groupby('vtd_tier', observed=True).agg(
    n_customers          =('customer_id', 'count'),
    total_revenue        =('total_revenue', 'sum'),
    avg_vtd              =('total_revenue', 'mean'),
    median_vtd           =('total_revenue', 'median'),
    avg_aov              =('aov', 'mean'),
    avg_aof              =('aof', 'mean'),
    repeat_rate_90d      =('repeat_purchase_90d', 'mean'),
    churn_rate_90d       =('churned_before_90d', 'mean'),
    pct_discount_acquired=('is_discount_acquired', 'mean'),
    pct_high_magnitude   =('is_high_magnitude', 'mean'),
    pct_subscriber       =('is_subscriber', 'mean'),
).reset_index()

tier_metrics['revenue_share'] = (
    tier_metrics['total_revenue'] / tier_metrics['total_revenue'].sum()
).round(4)

for col in ['avg_vtd', 'median_vtd', 'avg_aov', 'avg_aof']:
    tier_metrics[col] = tier_metrics[col].round(2)
for col in ['repeat_rate_90d', 'churn_rate_90d',
            'pct_discount_acquired', 'pct_high_magnitude', 'pct_subscriber']:
    tier_metrics[col] = tier_metrics[col].round(3)

tier_metrics = tier_metrics.sort_values('vtd_tier', ascending=False).reset_index(drop=True)
display(tier_metrics)

,vtd_tier,n_customers,total_revenue,avg_vtd,median_vtd,avg_aov,avg_aof,repeat_rate_90d,churn_rate_90d,pct_discount_acquired,pct_high_magnitude,pct_subscriber,revenue_share
0,D1: >1500,734,3453774.71,4705.42,2753.52,1143.51,7.63,0.580,0.420,0.059,0.012,0.069,0.5813
1,D2: 600–1500,990,909347.21,918.53,859.53,344.14,3.29,0.547,0.453,0.163,0.031,0.076,0.1531
2,D3: 350–600,1007,457427.77,454.25,444.90,238.64,1.69,0.469,0.531,0.279,0.063,0.053,0.0770
3,D4: 200–350,1667,438595.95,263.10,257.13,179.24,0.94,0.342,0.658,0.380,0.082,0.053,0.0738
4,D5: 150–200,1054,184396.02,174.95,174.00,139.49,0.53,0.244,0.756,0.399,0.063,0.049,0.0310
5,D6: 100–150,1712,212021.38,123.84,123.26,108.23,0.30,0.164,0.836,0.418,0.067,0.058,0.0357
6,D7: 75–100,1064,93931.66,88.28,88.91,80.28,0.21,0.126,0.874,0.342,0.082,0.038,0.0158
7,D8: 50–75,1829,114306.77,62.50,62.10,59.67,0.10,0.069,0.931,0.355,0.049,0.038,0.0192
8,D9: 25–50,1475,54510.65,36.96,36.90,35.15,0.10,0.077,0.923,0.384,0.064,0.010,0.0092
9,D10: <25,1406,22705.41,16.15,16.80,15.79,0.05,0.033,0.967,0.403,0.033,0.003,0.0038


---
## 4. Visualisation

### Step 4 — Four-panel decile decomposition chart

In [ ]:
# ── Step 4: Visualisation — 4 panel chart ────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

decile_labels = decile_metrics['revenue_decile'].astype(str)

# ── Panel 1: Revenue share by decile ─────────────────────────────────────────
ax1 = axes[0, 0]
bars = ax1.bar(decile_labels, decile_metrics['revenue_share'] * 100,
               color=COLORS['primary'], alpha=0.85, edgecolor='white')
for bar, val in zip(bars, decile_metrics['revenue_share']):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
             f'{val:.1%}', ha='center', fontsize=9, fontweight='bold')
ax1.set_title('Revenue Share by Decile', fontsize=13, fontweight='bold')
ax1.set_xlabel('Revenue Decile (D1=Lowest, D10=Highest)')
ax1.set_ylabel('% of Total Revenue')
ax1.yaxis.set_major_formatter(mtick.PercentFormatter())

# ── Panel 2: AOV and AOF by decile ───────────────────────────────────────────
ax2 = axes[0, 1]
ax2b = ax2.twinx()

x = range(len(decile_labels))
width = 0.4

bars_aov = ax2.bar([i - width/2 for i in x], decile_metrics['avg_aov'],
                   width=width, color=COLORS['primary'],
                   alpha=0.85, edgecolor='white', label='Avg AOV')
bars_aof = ax2b.bar([i + width/2 for i in x], decile_metrics['avg_aof'],
                    width=width, color=COLORS['accent'],
                    alpha=0.85, edgecolor='white', label='Avg AOF')

ax2.set_title('AOV and AOF by Revenue Decile', fontsize=13, fontweight='bold')
ax2.set_xlabel('Revenue Decile (D1=Lowest, D10=Highest)')
ax2.set_ylabel('Avg Order Value (SGD)', color=COLORS['primary'])
ax2b.set_ylabel('Avg Repeat Orders', color=COLORS['accent'])
ax2.set_xticks(list(x))
ax2.set_xticklabels(decile_labels)

lines1, labels1 = ax2.get_legend_handles_labels()
lines2, labels2 = ax2b.get_legend_handles_labels()
ax2.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

# ── Panel 3: 90-day repeat rate by decile ────────────────────────────────────
ax3 = axes[1, 0]
baseline_repeat = analysis['repeat_purchase_90d'].mean()

bars3 = ax3.bar(decile_labels, decile_metrics['repeat_rate_90d'] * 100,
                color=[COLORS['accent'] if r > baseline_repeat else COLORS['neutral']
                       for r in decile_metrics['repeat_rate_90d']],
                alpha=0.85, edgecolor='white')
ax3.axhline(baseline_repeat * 100, color=COLORS['secondary'], linestyle='--',
            linewidth=2, label=f'Baseline: {baseline_repeat:.1%}')
for bar, val in zip(bars3, decile_metrics['repeat_rate_90d']):
    ax3.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
             f'{val:.1%}', ha='center', fontsize=9, fontweight='bold')
ax3.set_title('90-Day Repeat Rate by Revenue Decile', fontsize=13, fontweight='bold')
ax3.set_xlabel('Revenue Decile (D1=Lowest, D10=Highest)')
ax3.set_ylabel('90-Day Repeat Rate (%)')
ax3.yaxis.set_major_formatter(mtick.PercentFormatter())
ax3.legend(fontsize=9)

# ── Panel 4: Discount acquisition rate by decile ─────────────────────────────
ax4 = axes[1, 1]
ax4b = ax4.twinx()

ax4.bar(decile_labels, decile_metrics['pct_discount_acquired'] * 100,
        color=COLORS['secondary'], alpha=0.7, edgecolor='white',
        label='Discount acquired')
ax4b.plot(decile_labels, decile_metrics['pct_subscriber'] * 100,
          color=COLORS['orange'] if 'orange' in COLORS else '#F97316',
          linewidth=2.5, marker='o', markersize=7, label='Subscriber rate')

ax4.set_title('Discount Acquisition & Subscriber Rate\nby Revenue Decile',
              fontsize=13, fontweight='bold')
ax4.set_xlabel('Revenue Decile (D1=Lowest, D10=Highest)')
ax4.set_ylabel('% Discount Acquired', color=COLORS['secondary'])
ax4b.set_ylabel('% Subscribers', color='#F97316')
ax4.yaxis.set_major_formatter(mtick.PercentFormatter())
ax4b.yaxis.set_major_formatter(mtick.PercentFormatter())

lines1, labels1 = ax4.get_legend_handles_labels()
lines2, labels2 = ax4b.get_legend_handles_labels()
ax4.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

plt.suptitle('LushProtein — Customer Revenue Decile Decomposition',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 5. Summary Findings

### Step 5 — Print summary findings

In [ ]:
# ── Step 5: Print summary findings ───────────────────────────────────────────
print("=" * 65)
print("DECILE DECOMPOSITION — KEY FINDINGS")
print("=" * 65)

top3 = decile_metrics.tail(3)
bot3 = decile_metrics.head(3)

print(f"\nTop 3 deciles (D8–D10):")
print(f"  Revenue share     : {top3['revenue_share'].sum():.1%}")
print(f"  Avg VTD           : SGD {top3['avg_vtd'].mean():.2f}")
print(f"  Avg AOV           : SGD {top3['avg_aov'].mean():.2f}")
print(f"  Avg AOF           : {top3['avg_aof'].mean():.1f} repeat orders")
print(f"  90d repeat rate   : {top3['repeat_rate_90d'].mean():.1%}")
print(f"  Discount acquired : {top3['pct_discount_acquired'].mean():.1%}")
print(f"  Subscriber rate   : {top3['pct_subscriber'].mean():.1%}")

print(f"\nBottom 3 deciles (D1–D3):")
print(f"  Revenue share     : {bot3['revenue_share'].sum():.1%}")
print(f"  Avg VTD           : SGD {bot3['avg_vtd'].mean():.2f}")
print(f"  Avg AOV           : SGD {bot3['avg_aov'].mean():.2f}")
print(f"  Avg AOF           : {bot3['avg_aof'].mean():.1f} repeat orders")
print(f"  90d repeat rate   : {bot3['repeat_rate_90d'].mean():.1%}")
print(f"  Discount acquired : {bot3['pct_discount_acquired'].mean():.1%}")
print(f"  Subscriber rate   : {bot3['pct_subscriber'].mean():.1%}")